# 08 — Pilot Report

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

The lender-pilot report: the champion against the incumbent (FICO midpoint) on the out-of-time vintage, with every exhibit a lender reads — discrimination with intervals, gains tables, the swap set at the lender's approval rate, calibration, segments, stability, and fair-lending tests on a proxy. The numbers come from `scorelab.report.pilot_exhibits`, the same function the MCP `run_pilot` tool wraps; the interpretation under each exhibit is yours.

Decisions: **Q — approval rate and segments**, **R — the fairness proxy**, **S — the interpretations and caveats**.

In [ ]:
frame = load_frame(include_indeterminate=True)
card, specs = S.load_card(MODELS / "champion")
for d in (frame,):
    sc = S.score(d, card, specs); d["score"] = sc["score"].values; d["eta"] = sc["eta"].values; d["p_bad"] = sc["p_bad"].values
tr = frame[(frame.split == "train") & frame.bad.notna()]; oot = frame[(frame.split == "oot") & frame.bad.notna()]
indet = frame[(frame.split == "oot") & frame.bad.isna()]
print("oot", oot.shape, "indeterminates in oot", len(indet))

### Decision Q — Approval rate and segments

The swap set is read at one operating point; use the lender's real approval rate (config default 70%). Segments: term, purpose, file depth (`total_acc` bands), vintage quarter.

**Your reasoning:**

_(write here)_

### Decision R — Fairness proxy

Lending Club carries no protected-class fields. Choose and document a proxy (a geography-based grouping from `addr_state` is the usual stand-in in public data), or state that fairness testing is out of scope for this data and keep the code path demonstrated.

**Your reasoning:**

_(write here)_

In [ ]:
APPROVAL_RATE = config.APPROVAL_RATE_FOR_SWAP
oot["file_depth"] = pd.cut(oot["total_acc"], [-1, 10, 20, 35, 1000], labels=["thin (<=10)", "11-20", "21-35", "thick (>35)"]).astype(str)
SEGMENTS = ("term_months", "file_depth", "purpose", "vintage_quarter")
SOUTH = {"TX","FL","GA","NC","SC","AL","MS","LA","TN","KY","AR","OK","VA","WV"}
oot["proxy_group"] = np.where(oot["addr_state"].isin(SOUTH), "south", "other")      # placeholder proxy: replace with your documented choice
GROUP_COL = "proxy_group"       # or None
ex = REP.pilot_exhibits(oot, tr["score"], approval_rate=APPROVAL_RATE, segment_cols=SEGMENTS, group_col=GROUP_COL, level="out-of-time (2016 vintage)")
display(ex["discrimination"]); print(ex["gini_lift"]); display(ex["swap_set"]["crosstab"]); print({k: v for k, v in ex["swap_set"].items() if k != "crosstab"})
display(ex["calibration_bands"]); print(ex["calibration_fit"]); print(ex["psi"])
for c, t in ex["segments"].items(): print(c); display(t)
if "fairness" in ex: display(ex["fairness"]["air"]); display(ex["fairness"]["smd"]); display(ex["fairness"]["performance"])

In [ ]:
# Evidence — where the indeterminates fall (the sensitivity the target definition promised)
if len(indet):
    display(pd.DataFrame({"group": ["bad", "good", "indeterminate"],
                          "n": [int((oot.bad == 1).sum()), int((oot.bad == 0).sum()), len(indet)],
                          "mean_score": [oot.loc[oot.bad == 1, "score"].mean(), oot.loc[oot.bad == 0, "score"].mean(), indet["score"].mean()]}).round(1))
    both = pd.concat([oot[["bad", "score"]], indet.assign(bad=1)[["bad", "score"]]]); g_as_bad = V.auc_gini_ks(both.bad, both.score)["gini"]
    both = pd.concat([oot[["bad", "score"]], indet.assign(bad=0)[["bad", "score"]]]); g_as_good = V.auc_gini_ks(both.bad, both.score)["gini"]
    print(f"OOT Gini excluding indeterminates {ex['discrimination'].loc[0, 'gini']:.4f} · scored as bad {g_as_bad:.4f} · scored as good {g_as_good:.4f}")

In [ ]:
# Optional — less-discriminatory-alternative search (refits the card without each attribute; a few minutes on full data)
RUN_LDA = False
if RUN_LDA and GROUP_COL:
    Wtr = B.transform_woe(tr, specs)
    def fit_and_score(attrs):
        c = S.fit_logistic(Wtr, tr["bad"], attrs); s = S.score(oot, c, specs); return s["score"].to_numpy(), s["eta"].to_numpy()
    display(F.lda_search(fit_and_score, card.attributes, oot["bad"], oot[GROUP_COL], APPROVAL_RATE))

### Decision S — Interpretations and caveats

One paragraph per exhibit, in the dictionary below, written by you. The caveats list must include the proxy-target statement (a single Lending Club loan with the delinquency date reconstructed from the last payment, against VantageScore's observed 90+ DPD on any tradeline), the booked-population statement, and the fairness proxy.

**Your text (edit the cell below):**

In [ ]:
INTERPRETATION = {
    "discrimination": "",
    "gains": "",
    "swap_set": "",
    "calibration": "",
    "segments": "",
    "stability": "",
    "fairness": "",
}
CAVEATS = [
    "Target is a proxy: one Lending Club loan per consumer, 90+ DPD reconstructed from the last payment date, 24 months from issue; "
    "VantageScore's definition observes 90+ DPD on any tradeline over 24 months from a bureau archive.",
    "Population is booked loans only (approved by Lending Club); see NB06 for the reject-inference sensitivity.",
    f"Fair-lending groups use a documented proxy ({GROUP_COL}); no protected-class data exists in the file.",
]
md_text = REP.assemble_pilot_report(ex, title="Pilot report: bureau-attribute scorecard vs FICO", lender="Lending Club (out-of-time 2016 vintage)",
                                    interpretation=INTERPRETATION, caveats=CAVEATS, path=config.DOCS_DIR / "pilot_report" / "pilot_report.md")
print(md_text[:1500]); print("...\nwritten to", config.DOCS_DIR / "pilot_report" / "pilot_report.md")